# Intro

## Global parameters

In [2]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [3]:
import os, pickle, platform, sys
import numpy as np
import copy
import torch

In [4]:
from collections import defaultdict

In [5]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [6]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [7]:
from scipy.stats import spearmanr

In [8]:
from tqdm.notebook import tqdm, trange

In [9]:
import datetime as dt

In [10]:
from bowtie import edges2bowtie

In [11]:
from tabulate import tabulate

### Home made

In [12]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [13]:
from auxiliary_functions import el2ks, bic

In [14]:
from sam_bowtie import block_and_fluxes as bnf

In [15]:
from bowtie_plot_functions import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from bowtie_plot_functions import _fdr as fdr

## Load data

In [16]:
DATA_FOLDER=HOME+'dati_elezioni/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'
MODELS_FOLDER='/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/benchmark_19/results_final/models/'

## Names for $\LaTeX$
#### Names as in the draft

In [17]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [18]:
cacca=parse_dico_names(GUARINO_FOLDER+'all_dico_labels.txt')

In [19]:
cacca['quirinale']

{5: 'journalists & Media',
 2: 'M5S',
 0: 'journalists & IV & Azione & +Europa & Media',
 4: 'Lega & FDI',
 1: 'PD',
 3: 'Media & journalists',
 6: 'FI'}

In [20]:
name_changer={'c0': 'Crisis - Centrists', 'c1': 'Crisis - Centre-right', 'c2': 'Crisis - M5S', 'c3': 'Crisis - PD', 'c4': 'Crisis - Media', 'e0': 'Elections - Centre-right', 'e1': 'Elections - Centre-left', 'e2': 'Elections - M5S', 'e3': 'Elections - Centrists', 'e4': 'Elections - Others', 'e5': 'Elections - Media', 'e6': 'Elections - Others 2', 'p0': 'President -  Centrists', 'p1': 'President -  PD', 'p2': 'President -  M5S', 'p3': 'President -  Media', 'p4': 'President -  Right', 'p5': 'President -  Others', 'p6': 'President -  FI'}

#### Proper labels for the Latex table

In [21]:
def sci_latex(x, dec=2, bold=False):
    # da Claude
    mant, exp = f'{x:.{dec}e}'.split('e')
    if bold:
        return f'$\\mathbf{{{mant}\\times 10^{{{int(exp)}}}}}$'
    return f'${mant}\\times 10^{{{int(exp)}}}$'


# Where are we?

## Get all (converged) solutions

In [22]:
solution_files=[f for f in os.listdir(MODELS_FOLDER) if f.endswith('.pkl')]
solution_files.sort()
solution_files

['crisi_dico0_decm_v070.pkl',
 'crisi_dico0_dwcm_v070.pkl',
 'crisi_dico0_qdecm_v070.pkl',
 'crisi_dico1_decm_v070.pkl',
 'crisi_dico1_dwcm_v070.pkl',
 'crisi_dico1_qdecm_v070.pkl',
 'crisi_dico2_decm_v070.pkl',
 'crisi_dico2_dwcm_v070.pkl',
 'crisi_dico2_qdecm_v070.pkl',
 'crisi_dico3_decm_v070.pkl',
 'crisi_dico3_dwcm_v070.pkl',
 'crisi_dico3_qdecm_v070.pkl',
 'crisi_dico4_decm_v070.pkl',
 'crisi_dico4_dwcm_v070.pkl',
 'crisi_dico4_qdecm_v070.pkl',
 'ita_elections_dico0_decm_v070.pkl',
 'ita_elections_dico0_dwcm_v070.pkl',
 'ita_elections_dico0_qdecm_v070.pkl',
 'ita_elections_dico1_decm_v070.pkl',
 'ita_elections_dico1_dwcm_v070.pkl',
 'ita_elections_dico1_qdecm_v070.pkl',
 'ita_elections_dico2_decm_v070.pkl',
 'ita_elections_dico2_dwcm_v070.pkl',
 'ita_elections_dico2_qdecm_v070.pkl',
 'ita_elections_dico3_decm_v070.pkl',
 'ita_elections_dico3_dwcm_v070.pkl',
 'ita_elections_dico3_qdecm_v070.pkl',
 'ita_elections_dico4_decm_v070.pkl',
 'ita_elections_dico4_dwcm_v070.pkl',
 'ita_ele

In [23]:
# Solutions were saved using a different version of numpy, which causes issues when loading them in the current environment. 
import importlib
import numpy.core as nc
sys.modules["numpy._core"] = nc
for sub in ("numeric", "multiarray", "_multiarray_umath", "numerictypes", "fromnumeric"):
    try:
        sys.modules["numpy._core." + sub] = importlib.import_module("numpy.core." + sub)
    except Exception:
        pass

In [24]:
final_dict={}
for solution in solution_files:
    # get the model type from the solution filename
    if 'qdecm' in solution.lower():
        model_type='qDECM'
    elif 'decm' in solution.lower():
        model_type='DECM'
    elif 'dwcm' in solution.lower():
        model_type='DWCM'
    else:
        print(f"Unknown model type in solution {solution}. Skipping.")
        continue

    # get the dataset from the solution filename
    if 'elections' in solution.lower():
        name='e'
    elif 'crisi' in solution.lower():
        name='c'
    elif 'quirinale' in solution.lower():
        name='p'
    else:
        print(f"Unknown dataset in solution {solution}. Skipping.")
        continue
    
    try:
        i_dico=solution.index('dico')
        dico=solution[i_dico+4]
    except ValueError:
        print(f"No 'dico' found in solution {solution}. Skipping.")
        continue

    dataset=f"{name}{dico}"
    if os.path.exists(MODELS_FOLDER+solution):
        try:
            with open(MODELS_FOLDER+solution, 'rb') as f:
                _temp = pickle.load(f)
        except Exception as e:
            print(f"{solution}: Error occurred while loading the solution: {e}. Deleting the file and skipping.")
            continue
        try:
            if _temp.sol.mre < 1e-5:  # Check if the model converged
                print(f"{solution}: Model converged.")
                if dataset not in final_dict:
                    final_dict[dataset]={}
                if model_type not in final_dict[dataset]:
                    final_dict[dataset][model_type]=_temp
                else:
                    print(f"{solution}: Converging model already exists in final_dict. Skipping.")
            else:
                print(f"{solution}: Model did not converge.")
        except AttributeError:
            print(f"{solution}: No 'sol' attribute in the loaded data.")

crisi_dico0_decm_v070.pkl: Model converged.
crisi_dico0_dwcm_v070.pkl: Model converged.
crisi_dico0_qdecm_v070.pkl: Model converged.
crisi_dico1_decm_v070.pkl: Model converged.
crisi_dico1_dwcm_v070.pkl: Model converged.
crisi_dico1_qdecm_v070.pkl: Model converged.
crisi_dico2_decm_v070.pkl: Model converged.
crisi_dico2_dwcm_v070.pkl: Model converged.
crisi_dico2_qdecm_v070.pkl: Model converged.
crisi_dico3_decm_v070.pkl: Model converged.
crisi_dico3_dwcm_v070.pkl: Model converged.
crisi_dico3_qdecm_v070.pkl: Model converged.
crisi_dico4_decm_v070.pkl: Model converged.
crisi_dico4_dwcm_v070.pkl: Model converged.
crisi_dico4_qdecm_v070.pkl: Model converged.
ita_elections_dico0_decm_v070.pkl: Model converged.
ita_elections_dico0_dwcm_v070.pkl: Model converged.
ita_elections_dico0_qdecm_v070.pkl: Model converged.
ita_elections_dico1_decm_v070.pkl: Model converged.
ita_elections_dico1_dwcm_v070.pkl: Model converged.
ita_elections_dico1_qdecm_v070.pkl: Model converged.
ita_elections_dico2_d

# BIC: DWCM vs. DECM

In [25]:
final_dict

{'c0': {'DECM': <dcms.models.decm.DECMModel at 0x17467a950>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174c2ee60>,
  'qDECM': <dcms.models.qdecm.qDECMModel at 0x174c2f2b0>},
 'c1': {'DECM': <dcms.models.decm.DECMModel at 0x174c2e650>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174be76d0>,
  'qDECM': <dcms.models.qdecm.qDECMModel at 0x174c2dd50>},
 'c2': {'DECM': <dcms.models.decm.DECMModel at 0x174c74910>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174be6fe0>,
  'qDECM': <dcms.models.qdecm.qDECMModel at 0x174c2f2e0>},
 'c3': {'DECM': <dcms.models.decm.DECMModel at 0x174c2d900>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174be5780>,
  'qDECM': <dcms.models.qdecm.qDECMModel at 0x174be78b0>},
 'c4': {'DECM': <dcms.models.decm.DECMModel at 0x174c760e0>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174c76860>,
  'qDECM': <dcms.models.qdecm.qDECMModel at 0x174c2dff0>},
 'e0': {'DECM': <dcms.models.decm.DECMModel at 0x174c76bc0>,
  'DWCM': <dcms.models.dwcm.DWCMModel at 0x174c74610>,
  'qDECM'

In [34]:
bic_list=[]
for key in tqdm(final_dict.keys(), desc="Calculating BICs"):
    _aux_vec=[key]
    for i_m, model in enumerate(['DWCM', 'DECM']):
        bt=final_dict[key][model].sol.best_theta
        _bic=final_dict[key][model].bic(bt)
        _aux_vec.append(_bic)

    # qDECM
    bt=final_dict[key]['qDECM'].sol.best_theta
    N=final_dict[key]['qDECM'].N
    _bic=final_dict[key]['qDECM'].bic(bt[:2*N],bt[2*N:])
    _aux_vec.append(_bic)
    
    bic_list.append(_aux_vec)

Calculating BICs:   0%|          | 0/19 [00:00<?, ?it/s]

### Table for $\LaTeX$

In [37]:
bic_list4latex_dwcm_decm=[]
for i in range(len(bic_list)):
    _aux_vec=[None]*3
    _aux_vec[0]=name_changer[bic_list[i][0]]
    for j in range(2):
        j_prime=1-j
        if bic_list[i][j+1]<bic_list[i][j_prime+1]:
            bold=True
        else:
            bold=False
        _aux_vec[j+1]=sci_latex(x=bic_list[i][j+1], dec=2, bold=bold)
    bic_list4latex_dwcm_decm.append(_aux_vec)


In [36]:
print(tabulate(bic_list4latex_dwcm_decm, headers=['DWCM', 'DECM'], tablefmt="latex_raw"))

\begin{tabular}{lll}
\hline
                          & DWCM                         & DECM                \\
\hline
 Crisis - Centrists       & $\mathbf{6.28\times 10^{6}}$ & $8.54\times 10^{6}$ \\
 Crisis - Centre-right    & $\mathbf{4.80\times 10^{6}}$ & $5.82\times 10^{6}$ \\
 Crisis - M5S             & $\mathbf{1.99\times 10^{6}}$ & $2.46\times 10^{6}$ \\
 Crisis - PD              & $\mathbf{5.21\times 10^{4}}$ & $8.77\times 10^{4}$ \\
 Crisis - Media           & $\mathbf{1.64\times 10^{5}}$ & $2.79\times 10^{5}$ \\
 Elections - Centre-right & $\mathbf{6.84\times 10^{6}}$ & $8.45\times 10^{6}$ \\
 Elections - Centre-left  & $\mathbf{1.01\times 10^{7}}$ & $1.46\times 10^{7}$ \\
 Elections - M5S          & $\mathbf{3.27\times 10^{6}}$ & $3.83\times 10^{6}$ \\
 Elections - Centrists    & $\mathbf{4.01\times 10^{6}}$ & $4.77\times 10^{6}$ \\
 Elections - Others       & $\mathbf{5.50\times 10^{5}}$ & $8.61\times 10^{5}$ \\
 Elections - Media        & $\mathbf{1.15\times 10^{4}}$ & $2.0

In [38]:
bic_list4latex_decm_qdecm=[]
for i in range(len(bic_list)):
    _aux_vec=[None]*3
    _aux_vec[0]=name_changer[bic_list[i][0]]
    for j in range(2):
        # j=0+2=2: DECM, j=1+2=3: qDECM
        j_prime=1-j
        if bic_list[i][j+2]<bic_list[i][j_prime+2]:
            bold=True
        else:
            bold=False
        _aux_vec[j+1]=sci_latex(x=bic_list[i][j+2], dec=2, bold=bold)
    bic_list4latex_decm_qdecm.append(_aux_vec)


In [40]:
print(tabulate(bic_list4latex_decm_qdecm, headers=['DECM', 'qDECM'], tablefmt="latex_raw"))

\begin{tabular}{lll}
\hline
                          & DECM                & qDECM                        \\
\hline
 Crisis - Centrists       & $8.54\times 10^{6}$ & $\mathbf{8.52\times 10^{6}}$ \\
 Crisis - Centre-right    & $5.82\times 10^{6}$ & $\mathbf{5.81\times 10^{6}}$ \\
 Crisis - M5S             & $2.46\times 10^{6}$ & $\mathbf{2.45\times 10^{6}}$ \\
 Crisis - PD              & $8.77\times 10^{4}$ & $\mathbf{8.77\times 10^{4}}$ \\
 Crisis - Media           & $2.79\times 10^{5}$ & $\mathbf{2.78\times 10^{5}}$ \\
 Elections - Centre-right & $8.45\times 10^{6}$ & $\mathbf{8.43\times 10^{6}}$ \\
 Elections - Centre-left  & $1.46\times 10^{7}$ & $\mathbf{1.45\times 10^{7}}$ \\
 Elections - M5S          & $3.83\times 10^{6}$ & $\mathbf{3.82\times 10^{6}}$ \\
 Elections - Centrists    & $4.77\times 10^{6}$ & $\mathbf{4.76\times 10^{6}}$ \\
 Elections - Others       & $8.61\times 10^{5}$ & $\mathbf{8.61\times 10^{5}}$ \\
 Elections - Media        & $2.00\times 10^{4}$ & $\mathbf{2.00

# Log-likelihood: the final showdown

In [41]:
ll_list=[]
for key in tqdm(_three_of_us):
    _aux_vec=[key]
    # DWCM
    bt=final_dict[key]['DWCM'].sol.best_theta
    dwcm_ll=final_dict[key]['DWCM'].neg_log_likelihood(bt)
    _aux_vec.append(dwcm_ll)
    # DECM
    bt=final_dict[key]['DECM'].sol.best_theta
    decm_ll=final_dict[key]['DECM'].neg_log_likelihood(bt)
    _aux_vec.append(decm_ll)
    # qDECM
    bt=final_dict[key]['qDECM'].sol.best_theta
    N=final_dict[key]['qDECM'].N
    qdecm_ll=final_dict[key]['qDECM'].neg_log_likelihood_strength(bt[:2*N], bt[2*N:])
    dcm_ll=final_dict[key]['qDECM']._dcm.neg_log_likelihood(bt[:2*N])
    qdecm_ll+=dcm_ll
    _aux_vec.append(qdecm_ll)


    ll_list.append(_aux_vec)

  0%|          | 0/19 [00:00<?, ?it/s]

In [42]:
ll_list

[['c0', 1849012.6624397475, 1683052.5131101352, 1677693.3291687602],
 ['c1', 1741343.9148470508, 1588878.6209270204, 1584077.508914125],
 ['c2', 703014.7964608616, 644545.5975686893, 642588.2869950536],
 ['c3', 7333.6382356581, 6459.069543821019, 6435.6741127125815],
 ['c4', 22538.885811110027, 20070.29079158493, 19970.44684593999],
 ['e0', 2331823.0879772, 2047094.1969169518, 2038734.4390023039],
 ['e1', 2569626.2228199365, 2317954.9752003415, 2308076.2060334026],
 ['e2', 1220620.318395807, 1080936.0908793015, 1077056.253483586],
 ['e3', 1426013.5037144632, 1230932.424297711, 1226703.8521784027],
 ['e4', 111044.35464076778, 102520.73283844037, 102265.6243969947],
 ['e5', 1326.8222701485802, 1166.0246480272356, 1161.913162102921],
 ['e6', 1009.551063314081, 829.3615667665814, 825.565706149297],
 ['p0', 589920.0661841637, 536817.8156254066, 535231.1997410976],
 ['p1', 93031.64831924175, 78970.0421161344, 78609.8550929483],
 ['p2', 503232.525596696, 459645.39416691073, 458194.834831436],

### Table for $\LaTeX$

In [43]:
ll_list4latex = copy.deepcopy(ll_list)

for i in range(len(ll_list4latex)):
    ll_list4latex[i][0]=name_changer[ll_list[i][0]]
    for j in range(3):
        
        if ll_list[i][j+1]==min(ll_list[i][1:]):
            bold=True
        else:
            bold=False
        ll_list4latex[i][j+1]=sci_latex(x=-ll_list[i][j+1], dec=3, bold=bold)


In [44]:
print(tabulate(ll_list4latex, headers=['DWCM', 'qDECM', 'DECM'], tablefmt="latex_raw"))

\begin{tabular}{llll}
\hline
                          & DWCM                  & qDECM                 & DECM                           \\
\hline
 Crisis - Centrists       & $-1.849\times 10^{6}$ & $-1.683\times 10^{6}$ & $\mathbf{-1.678\times 10^{6}}$ \\
 Crisis - Centre-right    & $-1.741\times 10^{6}$ & $-1.589\times 10^{6}$ & $\mathbf{-1.584\times 10^{6}}$ \\
 Crisis - M5S             & $-7.030\times 10^{5}$ & $-6.445\times 10^{5}$ & $\mathbf{-6.426\times 10^{5}}$ \\
 Crisis - PD              & $-7.334\times 10^{3}$ & $-6.459\times 10^{3}$ & $\mathbf{-6.436\times 10^{3}}$ \\
 Crisis - Media           & $-2.254\times 10^{4}$ & $-2.007\times 10^{4}$ & $\mathbf{-1.997\times 10^{4}}$ \\
 Elections - Centre-right & $-2.332\times 10^{6}$ & $-2.047\times 10^{6}$ & $\mathbf{-2.039\times 10^{6}}$ \\
 Elections - Centre-left  & $-2.570\times 10^{6}$ & $-2.318\times 10^{6}$ & $\mathbf{-2.308\times 10^{6}}$ \\
 Elections - M5S          & $-1.221\times 10^{6}$ & $-1.081\times 10^{6}$ & $\mathbf